# Spark Structured Streaming - Wikimedia Lab

This notebook consumes Wikipedia edit events from Kafka with Spark Structured Streaming.

Main explorations:
- bot vs human edit count
- average edit size
- tumbling windows
- overlapping windows
- producer-level filtering

> Run the Kafka environment and the Wikimedia producer before running this notebook.


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, from_unixtime,
    window, count, avg, abs
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, IntegerType, BooleanType
)


In [ ]:
# Spark 4.2.0 is used by the current pyspark-notebook Docker image.
spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0"
    ) \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")


## 1. Read the Kafka stream

Kafka stores the Wikimedia events in the `wikistreams` topic.

Spark reads this topic as a streaming DataFrame.


In [ ]:
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .load()


## 2. Parse the JSON messages

Kafka gives the message value as bytes.

I convert the JSON message into useful columns such as:
`title`, `user`, `timestamp`, `bot`, `minor` and `length`.


In [ ]:
simple_wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

parsed_df = raw_df.select(
    from_json(
        col("value").cast("string"),
        simple_wikimedia_schema
    ).alias("data")
).select("data.*")


## 3. First streaming aggregation: bots vs humans

I use a 1-hour window and count:
- bot edits (`bot = True`)
- human edits (`bot = False`)

This is a simple first example of an aggregation with Spark Structured Streaming.


In [ ]:
windowed_counts_df = parsed_df \
    .withColumn("event_time", from_unixtime(col("timestamp"))) \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

query_1 = windowed_counts_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

query_1.processAllAvailable()
query_1.stop()


## 4. Edit size analysis

The dataset gives the page size before and after an edit:

`edit_size = new size - old size`

A positive value means content was added.  
A negative value means content was removed.

I use `abs(edit_size)` because I want to measure how large the edit is,
not whether it added or removed content.


In [ ]:
edit_size_df = parsed_df \
    .withColumn(
        "event_time",
        from_unixtime(col("timestamp"))
    ) \
    .withColumn(
        "edit_size",
        col("length.new") - col("length.old")
    )

avg_edit_size_df = edit_size_df \
    .groupBy("bot") \
    .agg(
        avg(abs(col("edit_size"))).alias("avg_edit_size")
    )

query_2 = avg_edit_size_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

query_2.processAllAvailable()
query_2.stop()


### Result from my run

- bots modified about **431 characters** on average
- humans modified about **413 characters** on average

Bots made slightly larger edits on average in this sample.

The difference is small, so I cannot say that bots always make much bigger edits.


## 5. Tumbling window exploration

A tumbling window divides time into separate fixed blocks.

Example with 10-minute windows:

- 09:00 → 09:10
- 09:10 → 09:20
- 09:20 → 09:30

Each event belongs to only one window.


In [ ]:
tumbling_window_df = edit_size_df \
    .groupBy(
        window(col("event_time"), "10 minutes"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

query_3 = tumbling_window_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

query_3.processAllAvailable()
query_3.stop()


### Tumbling window result

In my run, for example:

- 09:50 → 10:00: humans = **154**, bots = **39**
- 10:00 → 10:10: humans = **151**, bots = **41**

The windows are separate, so each edit is counted in only one 10-minute block.


## 6. Overlapping window exploration

Now I use 10-minute windows with a new window every 5 minutes.

Example:

- 09:50 → 10:00
- 09:55 → 10:05
- 10:00 → 10:10

The windows overlap, so the same event can belong to more than one window.


In [ ]:
overlapping_window_df = edit_size_df \
    .groupBy(
        window(
            col("event_time"),
            "10 minutes",
            "5 minutes"
        ),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

query_4 = overlapping_window_df \
    .writeStream \
    .format("console") \
    .outputMode("complete") \
    .option("truncate", False) \
    .start()

query_4.processAllAvailable()
query_4.stop()


### Tumbling vs overlapping windows

**Tumbling window**
- separate time blocks
- one event belongs to one window

**Overlapping window**
- windows overlap
- one event can belong to several windows

Tumbling windows are easier to compare period by period.  
Overlapping windows give a more detailed view of changes over time.


## 7. Producer filtering exploration

The producer can filter Wikimedia events before sending them to Kafka.

French producer:

`server_name = fr.wikipedia.org`

English producer:

`server_name = en.wikipedia.org`

The English events are sent to another Kafka topic:

`wikistreams-en`

This shows that filtering can be done directly at the producer level.


## Final conclusion

In this lab, I used Kafka and Spark Structured Streaming to process Wikipedia edits.

I explored:
- bot and human edit counts
- average edit size
- tumbling windows
- overlapping windows
- producer-level filtering

The tumbling window divides the stream into separate time blocks.

The overlapping window allows one event to belong to several windows.

I also learned that the producer can filter the source before sending events to Kafka.
